# 🗣️ Fine-Tuning de TTS con Coqui (Tacotron2)

**Objetivo:** entrenar una voz sintética para una lengua de bajo recurso
(wolof, fula, bambara...) con **Coqui TTS** (Tacotron2 + vocoder), que es
**100% PyTorch**.

**Pipeline:**
1. Preparación del dataset (`metadata.csv`: `ruta|texto|locutor`)
2. (Opcional) Verificación de audio y espectrogramas mel
3. Configuración de entrenamiento (`config.json`)
4. Entrenamiento con el CLI `tts` de Coqui
5. Síntesis de audio con el modelo fine-tuneado + vocoder
6. Evaluación objetiva (duración, SNR, WER vía ASR)

**Requisitos:** Python 3.10+, `TTS` (Coqui), `torch`, `librosa`, `soundfile`,
`jiwer`.

> ℹ️ **Nota sobre TensorFlow:** la versión legacy usaba `TensorFlowTTS`, que
> está obsoleto. Este notebook usa exclusivamente Coqui TTS (PyTorch).
>
> 🔑 **Modelos gated:** si algún modelo requiere autenticación, define
> `export HF_TOKEN=hf_...` en el entorno. Nunca se hardcodean tokens.

---
## 0. Instalación

```bash
pip install TTS torch librosa soundfile jiwer
sudo apt-get install -y ffmpeg
```

Coqui TTS instalará sus dependencias PyTorch automáticamente. Si hay
conflictos con `numpy`/`pandas`, reinstala las versiones compatibles:

```bash
pip install "numpy<2" "pandas<2"
```

In [ ]:
# === IMPORTS Y DISPOSITIVO ===
import os
import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"HF_TOKEN definido: {bool(os.environ.get('HF_TOKEN'))}")

---
## 1. Configuración

Estructura de datos esperada:

```
DATA_DIR/
├── audio/          # .wav de un solo locutor (o subcarpetas por locutor)
└── metadata.csv    # ruta|texto|locutor
```

In [ ]:
# === CONFIGURACIÓN ===
DATA_DIR = Path("./data/tts")          # <- CAMBIA ESTO
OUTPUT_DIR = Path("./models/tts_coqui")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    # --- Audio ---
    sampling_rate: int = 22050
    n_mels: int = 80
    hop_length: int = 256
    win_length: int = 1024

    # --- Dataset ---
    audio_dir: str = "audio"            # subcarpeta dentro de DATA_DIR
    speaker: str = "locutor1"
    metadata_file: str = "metadata.csv"

    # --- Entrenamiento (Coqui) ---
    model_name: str = "tts_models/en/ljspeech/tacotron2-DDC"  # base preentrenada
    epochs: int = 100
    batch_size: int = 16
    learning_rate: float = 1e-3
    num_workers: int = 2


config = Config()
print(config)

---
## 2. Preparación del dataset (`metadata.csv`)

Coqui espera un `metadata.csv` con formato `ruta|texto|locutor` (delimitador
`|`). `build_metadata()` lo genera automáticamente:

- Recorre `DATA_DIR/audio/` buscando `.wav`
- Usa `transcriptions.txt` si existe (una línea por audio, en el mismo orden),
  o el nombre del fichero si no hay transcripción
- Escribe `DATA_DIR/metadata.csv`

> ⚠️ Calidad de datos > cantidad: 30–60 minutos de audio limpio por locutor
> dan resultados mucho mejores que horas con ruido.

In [ ]:
# === GENERAR metadata.csv ===
def build_metadata(data_dir: Path, audio_dir: str, speaker: str,
                   metadata_file: str = "metadata.csv") -> Path:
    """Construye metadata.csv (ruta|texto|locutor) desde la carpeta de audio."""
    data_dir, audio_dir = Path(data_dir), Path(data_dir) / audio_dir
    metadata_path = data_dir / metadata_file

    # Transcripciones opcionales: una línea por audio, en orden alfabético
    transcripts_path = data_dir / "transcriptions.txt"
    transcripts = None
    if transcripts_path.exists():
        with open(transcripts_path, encoding="utf-8") as f:
            transcripts = [l.strip() for l in f if l.strip()]

    wavs = sorted(audio_dir.glob("*.wav"))
    if not wavs:
        raise FileNotFoundError(f"No hay .wav en {audio_dir}")

    rows = []
    for i, wav in enumerate(wavs):
        text = transcripts[i] if transcripts else wav.stem.replace("_", " ")
        rows.append(f"{wav}|{text}|{speaker}")

    with open(metadata_path, "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")

    print(f"✅ {len(rows)} muestras -> {metadata_path}")
    return metadata_path


metadata_path = build_metadata(DATA_DIR, config.audio_dir, config.speaker,
                               config.metadata_file)

---
## 3. (Opcional) Verificación de audio y espectrogramas mel

Comprueba duraciones y visualiza un espectrograma mel de muestra. También
permite descartar audios corruptos o demasiado cortos/largos.

In [ ]:
# === VERIFICACIÓN DE AUDIO ===
import librosa
import matplotlib.pyplot as plt
import librosa.display


def audio_stats(data_dir: Path, audio_dir: str, sr=22050):
    """Estadísticas de duración de los audios del dataset."""
    durations = []
    for wav in sorted((Path(data_dir) / audio_dir).glob("*.wav")):
        try:
            y, _ = librosa.load(wav, sr=sr)
            durations.append(len(y) / sr)
        except Exception as e:
            print(f"⚠️ Corrupto: {wav.name} ({e})")
    durations = np.array(durations)
    print(f"Audios: {len(durations)} | "
          f"min={durations.min():.1f}s media={durations.mean():.1f}s "
          f"max={durations.max():.1f}s" if len(durations) else "Sin audios")
    return durations


durations = audio_stats(DATA_DIR, config.audio_dir, config.sampling_rate)


def plot_mel(audio_path: Path, config: Config):
    """Espectrograma mel de un audio de muestra."""
    y, sr = librosa.load(audio_path, sr=config.sampling_rate)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=config.n_mels,
        hop_length=config.hop_length, win_length=config.win_length)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_db, sr=sr, hop_length=config.hop_length,
                             x_axis="time", y_axis="mel")
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"Mel espectrograma: {audio_path.name}")
    plt.tight_layout()
    plt.show()


# plot_mel(next((Path(DATA_DIR) / config.audio_dir).glob("*.wav")))  # <- descomenta

---
## 4. Configuración de entrenamiento (`config.json`)

Coqui entrena con un fichero `config.json` que describe modelo, dataset y
trainer. Aquí se genera para **Tacotron2** con los parámetros del paso 1.

> Para **XTTS-v2** (mejor calidad con pocos datos), consulta la alternativa al
> final del notebook.

In [ ]:
# === GENERAR config.json ===
max_mel_length = int(max(durations) * config.sampling_rate / config.hop_length) + 1

train_config = {
    "model": {
        "name": "Tacotron2",
        "path": "tacotron2",
        "config": {
            "n_mel_channels": config.n_mels,
            "symbols_embedding_dim": 512,
            "encoder_conv_filters": 256,
            "encoder_conv_kernel_size": 5,
            "encoder_lstm_units": 256,
            "decoder_lstm_units": 256,
            "max_decoder_steps": max_mel_length,
            "dropout_rate": 0.5,
        },
    },
    "dataset": {
        "name": "CustomDataset",
        "path": str(DATA_DIR),
        "meta_file_train": config.metadata_file,
        "meta_file_val": config.metadata_file,
        "text_cleaners": ["basic_cleaners"],
        "max_wav_value": 32768.0,
        "sampling_rate": config.sampling_rate,
        "filter_length": config.win_length,
        "hop_length": config.hop_length,
        "win_length": config.win_length,
        "n_mel_channels": config.n_mels,
        "mel_fmin": 0.0,
        "mel_fmax": 8000.0,
    },
    "trainer": {
        "epochs": config.epochs,
        "learning_rate": config.learning_rate,
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "checkpoint_path": str(OUTPUT_DIR / "checkpoints"),
        "log_path": str(OUTPUT_DIR / "logs"),
        "save_step": 1000,
        "save_best_after": 1000,
        "print_step": 50,
        "plot_step": 100,
        "run_eval": True,
        "run_test": True,
    },
}

config_path = DATA_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(train_config, f, indent=2)
print(f"✅ Configuración guardada en {config_path}")

---
## 5. Entrenamiento

Coqui entrena desde su CLI (el entrenamiento es 100% PyTorch):

```bash
tts --config_path data/config.json --train
```

El checkpoint del mejor modelo se guarda en
`models/tts_coqui/checkpoints/best_model.pth.tar`.

> ⏱️ En CPU, reduce `epochs` a 10–20 y `batch_size` a 4 para una prueba
> rápida. El entrenamiento real requiere GPU (8 GB+).

In [ ]:
# === ENTRENAMIENTO (CLI de Coqui TTS) ===
# !tts --config_path data/config.json --train
# (descomenta la línea anterior; ejecuta en terminal para ver el progreso)
print("Ejecuta en terminal:")
print(f"  tts --config_path {DATA_DIR / 'config.json'} --train")

---
## 6. Síntesis de audio

Carga el modelo entrenado (Tacotron2) + un **vocoder** (HiFi-GAN) y genera
audio desde texto. El vocoder convierte los mel espectrogramas en forma de onda.

In [ ]:
# === SÍNTESIS ===
from TTS.api import TTS

# Ruta al mejor checkpoint entrenado (ajusta si es distinta)
best_model_path = OUTPUT_DIR / "checkpoints" / "best_model.pth.tar"

# Vocoder HiFi-GAN (preentrenado, calidad estándar)
vocoder_name = "vocoder_models/en/ljspeech/hifigan_v2"

tts = TTS(model_path=str(best_model_path),
          config_path=str(DATA_DIR / "config.json"),
          vocoder_path=vocoder_name,
          progress_bar=True, gpu=torch.cuda.is_available())


def synthesize(text: str, out_path: str = "output.wav") -> str:
    """Genera audio desde texto y lo guarda en WAV."""
    tts.tts_to_file(text=text, file_path=out_path)
    print(f"✅ Audio generado: {out_path}")
    return out_path


# --- Ejemplo (cambia el texto a la lengua de tu dataset) ---
sample_text = "Jàmm ngën, nanga def?"   # "Hola, ¿cómo estás?" en wolof
synthesize(sample_text, str(OUTPUT_DIR / "muestra.wav"))

---
## 7. Evaluación objetiva

Métricas sin evaluadores humanos:
1. **Duración** del audio generado (vs. longitud del texto)
2. **SNR** si hay un audio de referencia
3. **WER** (word error rate): se transcribe el audio generado con **Whisper**
   y se compara con el texto original — mide la inteligibilidad.

> La calidad percibida (naturalidad) se evalúa con **MOS** (Mean Opinion
> Score): 5 oyentes puntúan de 1 a 5; media ≥ 4 es excelente.

In [ ]:
# === EVALUACIÓN ===
from jiwer import wer


def evaluate_tts(generated_audio: str, text: str = None,
                 reference_audio: str = None) -> dict:
    """Métricas objetivas del audio TTS generado."""
    metrics = {}
    print("📊 Evaluación TTS")
    print("=" * 40)

    # 1. Duración
    audio, sr = sf.read(generated_audio)
    duration = len(audio) / sr
    metrics["duration_s"] = round(duration, 2)
    print(f"  Duración: {duration:.2f}s")

    # 2. SNR contra referencia (si existe)
    if reference_audio:
        ref, sr_ref = sf.read(reference_audio)
        min_len = min(len(audio), len(ref))
        signal, noise = audio[:min_len], ref[:min_len] - audio[:min_len]
        snr = 10 * np.log10(np.var(signal) / (np.var(noise) + 1e-10))
        metrics["snr_db"] = round(snr, 2)
        print(f"  SNR: {snr:.2f} dB")

    # 3. WER vía Whisper (inteligibilidad)
    if text:
        import whisper
        asr = whisper.load_model("small")
        transcribed = asr.transcribe(generated_audio)["text"].strip()
        wer_score = wer([text.lower()], [transcribed.lower()])
        metrics["wer"] = round(wer_score, 4)
        print(f"  Texto original : {text}")
        print(f"  ASR transcrito : {transcribed}")
        print(f"  WER: {wer_score:.4f} (menor = mejor)")

    print("=" * 40)
    return metrics


evaluate_tts(str(OUTPUT_DIR / "muestra.wav"), text=sample_text)

---
## 8. Alternativa: XTTS-v2 (mejor calidad con pocos datos)

Si la lengua tiene **pocos minutos** de audio (3–10 min), **XTTS-v2** de Coqui
da mejor calidad que Tacotron2. Fine-tuning:

```bash
tts --model_name tts_models/multilingual/multi-dataset/xtts_v2 \
    --dataset_audio_metadata data/metadata_xtts.csv \
    --output_path models/tts_xtts --num_epochs 100 --batch_size 16
```

Con `metadata_xtts.csv` en formato `ruta|texto|locutor` (el mismo del paso 2).
Inferencia con voz de referencia:

```python
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
tts.tts_to_file(text=sample_text, speaker_wav="ref.wav",
                language="wol", file_path="xtts_out.wav")
```

---

## Resumen

| Paso | Qué hace | Salida |
|---|---|---|
| 2 | Dataset + `metadata.csv` | `data/metadata.csv` |
| 3 | Verificación de audio/mel | estadísticas, figura |
| 4 | Config de entrenamiento | `data/config.json` |
| 5 | Entrenamiento Coqui (PyTorch) | `models/tts_coqui/checkpoints/` |
| 6 | Síntesis con vocoder | `models/tts_coqui/muestra.wav` |
| 7 | Evaluación objetiva | duración, SNR, WER |